# South Sudan Tabular Data

This notebook is used to prepare the location data to upload to Strapi.

The data model is as follows:
```typescript
interface Location {
  name: string; // Required
  type: 'administrative' | 'hydrological'; // Required
  level: number; // Required, must be 1, 2 or 3
  code: string; // Required, reference to the layer geometry
  parent: Location; // 1-to-1 relation to a parent location
}
```

The data can be exported as a JSON file with the following structure:
```json
{
  "version": 2,
  "data": {
    "api::location.location": {
      "1": {
        "id": 1,
        "name": "Level 1 example",
        "type": "administrative",
        "level": 1,
        "code": "01",
        "createdAt": "2024-10-28T13:40:23.054Z",
        "updatedAt": "2024-10-28T13:41:03.092Z",
        "parent": null,
        "createdBy": null,
        "updatedBy": null
      },
      "2": {
        "id": 2,
        "name": "Level 2 example",
        "type": "administrative",
        "level": 2,
        "code": "011",
        "createdAt": "2024-10-28T13:40:36.041Z",
        "updatedAt": "2024-10-28T13:40:59.185Z",
        "parent": 1,
        "createdBy": null,
        "updatedBy": null
      },
      "3": {
        "id": 3,
        "name": "Level 3 example",
        "type": "administrative",
        "level": 3,
        "code": "012",
        "createdAt": "2024-10-28T13:40:50.463Z",
        "updatedAt": "2024-10-28T13:40:50.463Z",
        "parent": 1,
        "createdBy": null,
        "updatedBy": null
      }
    }
  }
}
```




## Setup

### Library import

In [1]:
# imports
import json
import sys
from datetime import datetime
from pprint import pprint

# Include local library paths if you have ../src/utils.py
sys.path.append("../src/")
sys.path.append("../src/animations")
sys.path.append("../src/datasets")
sys.path.append("../src/datasets/factory")
sys.path.append("../src/helpers")

from datasets.datasets import dataset_database

## Dataset information

In [2]:
datasets = dataset_database.datasets()
pprint(datasets)

{'Agricultural drought exposure': <datasets.datasets.Dataset object at 0x7f2042d9fef0>,
 'Agricultural drought hazard': <datasets.datasets.Dataset object at 0x7f2042d9fe90>,
 'Boundaries': <datasets.datasets.Dataset object at 0x7f2042d9ff20>,
 'Contextual layers': <datasets.datasets.Dataset object at 0x7f2042d9ff50>,
 'EO-based flood exposure': <datasets.datasets.Dataset object at 0x7f2042d9ff80>,
 'EO-based flood hazard': <datasets.datasets.Dataset object at 0x7f2042d9ffb0>,
 'Hydrographic data': <datasets.datasets.Dataset object at 0x7f2042d9ffe0>,
 'Hydrometeorological Data': <datasets.datasets.Dataset object at 0x7f2042d9fb60>,
 'Meteorological drought exposure': <datasets.datasets.Dataset object at 0x7f2042d9fb30>,
 'Meteorological drought hazard': <datasets.datasets.Dataset object at 0x7f2042d9fb00>,
 'Model-based flood exposure': <datasets.datasets.Dataset object at 0x7f2118117aa0>,
 'Model-based flood hazard': <datasets.datasets.Dataset object at 0x7f2118117980>,
 'Populated in

## Load Layers

In [3]:
dataset = datasets["Boundaries"]
layers = dataset.layers()
layers

{'Administrative Boundaries - adm0': <datasets.datasets.Layer at 0x7f2073b67fb0>,
 'Administrative Boundaries - adm1': <datasets.datasets.Layer at 0x7f2073b677a0>,
 'Administrative Boundaries - adm2': <datasets.datasets.Layer at 0x7f2073b67dd0>,
 'Administrative Boundaries - adm3': <datasets.datasets.Layer at 0x7f2073b67c20>,
 'Hydrological Basins': <datasets.datasets.Layer at 0x7f2042dd44a0>}

## Process Data

In [7]:
layer_types = {
    "Administrative Boundaries - adm1": "administrative",
    "Administrative Boundaries - adm2": "administrative",
    "Administrative Boundaries - adm3": "administrative",
    "Hydrological Basins": "hydrological",
}

code_column = {
    "Administrative Boundaries - adm1": "adm1_pcode",
    "Administrative Boundaries - adm2": "adm2_pcode",
    "Administrative Boundaries - adm3": "adm3_pcode",
    "Hydrological Basins": "objectid",
}

name_column = {
    "Administrative Boundaries - adm1": "adm1_en",
    "Administrative Boundaries - adm2": "adm2_en",
    "Administrative Boundaries - adm3": "adm3_en",
    "Hydrological Basins": "basinname",
}

In [8]:
# Initialize the JSON structure
json_data = {"version": 2, "data": {"api::location.location": {}}}

# Dictionary to keep track of parent IDs
parent_ids = {}

location_id = 1
for layer_name, layer_type in layer_types.items():
    print(f"Processing {layer_name}")
    layer = layers[layer_name]
    df = layer.get_data()

    for _index, row in df.iterrows():
        code = str(row[code_column[layer_name]])
        if "Administrative Boundaries" in layer_name:
            level = int(layer_name.split("adm")[-1]) if "adm" in layer_name else 1
            parent_ids[code] = location_id
            if level > 1:
                try:
                    parent_id = parent_ids[code[:-2]]
                except KeyError:
                    parent_id = None
            else:
                parent_id = None
        else:
            level = 1
            parent_id = None

        location_data = {
            "id": location_id,
            "name": row[name_column[layer_name]],
            "type": layer_type,
            "level": level,
            "code": code,
            "createdAt": datetime.now().isoformat(),
            "updatedAt": datetime.now().isoformat(),
            "parent": parent_id,
            "createdBy": None,
            "updatedBy": None,
        }
        json_data["data"]["api::location.location"][str(location_id)] = location_data
        location_id += 1

# Convert to JSON string
json_string = json.dumps(json_data, indent=2)

# Write to file
with open("../data/processed/locations.json", "w") as f:
    f.write(json_string)

Processing Administrative Boundaries - adm1
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm1_4326_SouthSudan_20230829_20240228.shp...
Processing Administrative Boundaries - adm2
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm2_4326_SouthSudan_20230829_20240228.shp...
Processing Administrative Boundaries - adm3
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Ancillary%20Layers/OCHA-SubnationalAdministrativeBoundaries/WBHYDROSSD_OCHA_SubnationalAdministrativeBoundaries-adm3_4326_SouthSudan_20230829_20240228.shp...
Processing Hydrological Basins
Loading data from https://storage.googleapis.com/wbhydross_deliverables/D3-Database/00-%20Anc